# MFFT Training — Step by Step
**Multi-Frequency Fusion Transformer for AI Image Detection**

In [ ]:
# Cell 1: Imports & Setup
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

# Fix project root
PROJECT_ROOT = Path(os.path.abspath('')).parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
print(f'Project root: {PROJECT_ROOT}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')


In [ ]:
# Cell 2: Load Dataset
from src.dataset import AIDetectionDataset, ImageTransform, create_dataloaders
from src.config import Config

# Use base model
cfg = Config()
cfg.training.model_variant = 'base'
cfg.training.image_size = 224
cfg.training.batch_size = 8
cfg.training.mixed_precision = False
cfg.training.num_workers = 0
cfg.training.epochs = 1
cfg.training.val_check_interval = 50
cfg.training.gradient_accumulation_steps = 1
cfg.dataset.val_split = 0.1
cfg.dataset.test_split = 0.1

# Load full dataset
full_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT),
    metadata_paths=[str(PROJECT_ROOT / 'dataset' / 'metadata' / 'all.csv')],
    transform=None,
    is_train=True,
    size=cfg.training.image_size,
    undersample=True,
)

print(f'Total samples: {len(full_dataset)}')
print(f'Real: {sum(1 for _, l in full_dataset.samples if l==0)}')
print(f'AI:   {sum(1 for _, l in full_dataset.samples if l==1)}')

In [ ]:
# Cell 3: Split into Train/Val/Test
from sklearn.model_selection import train_test_split

labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(
    indices, test_size=cfg.dataset.val_split + cfg.dataset.test_split,
    stratify=labels, random_state=42,
)

temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=cfg.dataset.test_split / (cfg.dataset.val_split + cfg.dataset.test_split),
    stratify=temp_labels, random_state=42,
)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

# Build datasets with transforms
train_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=True),
    is_train=True, size=cfg.training.image_size, undersample=False,
)
train_dataset.samples = [full_dataset.samples[i] for i in train_idx]

val_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
val_dataset.samples = [full_dataset.samples[i] for i in val_idx]

test_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
test_dataset.samples = [full_dataset.samples[i] for i in test_idx]

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

In [ ]:
# Cell 4: Build Model
from src.model import build_mfft, count_parameters, MFFTWithExplainability

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = build_mfft('base')
model = model.to(device)
print(f'Parameters: {count_parameters(model):,}')

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
print('Model ready')

In [ ]:
# Cell 5: Train 1 Epoch (watch progress)
model.train()
total_loss = 0
correct = 0
total = 0
start = time.time()

pbar = tqdm(train_loader, desc='Training')
for batch_idx, (images, labels) in enumerate(pbar):
    images, labels = images.to(device), labels.to(device)
    
    logits = model(images)
    loss = criterion(logits, labels)
    loss.backward()
    
    optimizer.step()
    optimizer.zero_grad()
    
    total_loss += loss.item()
    preds = logits.argmax(dim=-1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)
    
    if (batch_idx + 1) % 20 == 0:
        acc = correct / total * 100
        pbar.set_postfix({'loss': f'{total_loss/(batch_idx+1):.4f}', 'acc': f'{acc:.2f}%'})

elapsed = time.time() - start
print(f'Epoch done in {elapsed:.1f}s')
print(f'Train loss: {total_loss/len(train_loader):.4f}, acc: {correct/total*100:.2f}%')

In [ ]:
# Cell 6: Validate
model.eval()
val_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Validating'):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        val_loss += loss.item()
        preds = logits.argmax(dim=-1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

print(f'Val loss: {val_loss/len(val_loader):.4f}, acc: {val_correct/val_total*100:.2f}%')

In [ ]:
# Cell 7: Full Training Loop (adjust epochs)
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

NUM_EPOCHS = 30  # adjust as needed; 20-50 recommended

# Track training history for Figure 2
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

# Track training history for Figure 2
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

model = build_mfft('base').to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS * len(train_loader), T_mult=2, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

best_acc = 0
for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': f'{total_loss/(total/cfg.training.batch_size):.4f}', 'acc': f'{correct/total*100:.2f}%', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
    
    train_acc = correct / total * 100
    history["train_acc"].append(train_acc)
    history["train_loss"].append(total_loss / len(train_loader))
    history["train_acc"].append(train_acc)
    history["train_loss"].append(total_loss / len(train_loader))
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total * 100
    history["val_acc"].append(val_acc)
    history["val_loss"].append(val_loss / len(val_loader))
    history["val_acc"].append(val_acc)
    history["val_loss"].append(val_loss / len(val_loader))
    print(f'Epoch {epoch+1}: train={train_acc:.2f}%, val={val_acc:.2f}%')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), PROJECT_ROOT / 'model' / 'checkpoints' / 'best_mfft_base.pt')
        print(f'  Saved best model ({best_acc:.2f}%)')

print(f'\nBest val accuracy: {best_acc:.2f}%')

In [ ]:
# Cell 8: Save Final Model
model.eval()
torch.save(model.state_dict(), PROJECT_ROOT / 'model' / 'checkpoints' / 'mfft_base_final.pt')
print('Model saved to model/checkpoints/mfft_base_final.pt')
print(f'File size: {os.path.getsize(PROJECT_ROOT / "model" / "checkpoints" / "mfft_base_final.pt") / 1e6:.1f} MB')

In [ ]:
# Cell 9: Test Prediction on a Sample (from test set)
img_path = test_dataset.samples[0][0]
img = Image.open(img_path).convert('RGB')
transform = ImageTransform(size=cfg.training.image_size, augment=False)
tensor = transform(img).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    logits = model(tensor)
    probs = F.softmax(logits, dim=-1)

pred = 'AI-generated' if probs[0][1] > probs[0][0] else 'Real'
print(f'Image: {os.path.basename(img_path)}')
print(f'Real prob: {probs[0][0]:.4f}')
print(f'AI prob:   {probs[0][1]:.4f}')
print(f'Prediction: {pred}')

---
## Manuscript Figures (Cell 10)
Generate all publication-quality figures after training.

In [ ]:
# Cell 10: Generate All Manuscript Figures (on held-out test set)
from src.visualize import generate_all_figures
from sklearn.metrics import confusion_matrix
import numpy as np

FIGS_DIR = PROJECT_ROOT / 'paper' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# Evaluate on held-out test set for paper results
all_test_labels = []
all_test_probs = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=-1)
        all_test_labels.extend(labels.cpu().numpy())
        all_test_probs.extend(probs[:, 1].cpu().numpy())

y_true = np.array(all_test_labels)
y_score = np.array(all_test_probs)
y_pred = (y_score >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred)

# Class labels from dataset
all_labels = [s[1] for s in full_dataset.samples]

# Pick a sample image for frequency decomposition
sample_img = None
for cls_dir in ['real', 'ai_generated', 'ai_altered']:
    d = PROJECT_ROOT / 'dataset' / 'images' / cls_dir
    if d.exists():
        files = list(d.glob('*.jpg')) or list(d.glob('*.png'))
        if files:
            sample_img = str(files[0])
            break

# Generate all 10 figures (use test_loader for sample predictions + heatmaps)
paths = generate_all_figures(
    history=history,
    train_loader=train_loader,
    history=history,
    train_loader=train_loader,
    model=model,
    val_loader=test_loader,
    device=device,
    y_true=y_true,
    y_score=y_score,
    cm=cm,
    labels=all_labels,
    sample_image_path=sample_img,
    output_dir=FIGS_DIR,
)

print('\n' + '='*60)
print('All manuscript figures saved to paper/figures/')
for name, path in paths.items():
    print(f'  {name}: {path.name}')
print('='*60)

In [ ]:
# Cell 11: Per-Category Accuracy Breakdown
from collections import defaultdict
from pathlib import Path

# Determine original category from file path
category_map = defaultdict(list)
for idx, (img_path, true_label) in enumerate(test_dataset.samples):
    parent_dir = Path(img_path).parent.name
    if parent_dir == "real":
        category = "Real"
    elif parent_dir == "ai_generated":
        category = "AI Generated"
    elif parent_dir == "ai_altered":
        category = "AI Altered"
    else:
        category = "Unknown"
    category_map[category].append((y_pred[idx] == true_label, y_score[idx], true_label, y_pred[idx]))

print("=" * 65)
print(f"{'Category':<20} {'Count':>8} {'Accuracy':>10} {'Avg Conf':>10} {'AUC':>8}")
print("-" * 65)
from sklearn.metrics import roc_auc_score
overall_correct = 0
overall_total = 0
rows = []
for cat in ["Real", "AI Generated", "AI Altered", "Unknown"]:
    if cat not in category_map:
        continue
    items = category_map[cat]
    correct_list = [c for c, _, _, _ in items]
    scores = [s for _, s, _, _ in items]
    true_vs_pred = [(t, p) for _, _, t, p in items]
    n = len(correct_list)
    acc = sum(correct_list) / n * 100
    avg_conf = sum(scores) / n * 100
    try:
        auc = roc_auc_score([t for t, _ in true_vs_pred], [p for _, p in true_vs_pred])
    except Exception:
        auc = 0.0
    rows.append((cat, n, acc, avg_conf, auc))
    overall_correct += sum(correct_list)
    overall_total += n
    print(f"{cat:<20} {n:>8} {acc:>9.2f}% {avg_conf:>9.2f}% {auc:>7.4f}")

print("-" * 65)
overall_acc = overall_correct / overall_total * 100
print(f"{'OVERALL':<20} {overall_total:>8} {overall_acc:>9.2f}%")
print("=" * 65)

# Save to table
import json
tables_dir = PROJECT_ROOT / "paper" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
with open(tables_dir / "per_category_accuracy.json", "w") as f:
    import numpy as np
    def _to_py(x):
        return float(x) if isinstance(x, (np.floating, np.integer)) else x
    json.dump({
        "categories": {r[0]: {"count": int(r[1]), "accuracy": round(float(r[2]), 2), "avg_confidence": round(float(r[3]), 2), "auc": round(float(r[4]), 4)} for r in rows},
        "overall": {"count": int(overall_total), "accuracy": round(float(overall_acc), 2)}
    }, f, indent=2, default=_to_py)
print(f"Saved to {tables_dir / 'per_category_accuracy.json'}")
